In [ ]:
594 / 2

In [ ]:
data_folder = "/home/jbeckwith/Documents/Dropbox/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Data/Salix/CS505CU_Calibration"
gain = IO.read_tiff(os.path.join(data_folder, "gain.tif"))
offset = IO.read_tiff(os.path.join(data_folder, "offset.tif"))
variance = IO.read_tiff(os.path.join(data_folder, "variance.tif"))
readnoise = IO.read_tiff(os.path.join(data_folder, "readnoise.tif"))
rqe = IO.read_tiff(os.path.join(data_folder, "rqe.tif"))

In [ ]:
folder = "Spectra/Em/Dyes/"
emission_spectra = np.sort(os.listdir(folder))
ATTO390_name = emission_spectra[19]
ATTO488_name = emission_spectra[22]
ATTO550_name = emission_spectra[28]
ATTO647N_name = emission_spectra[36]
ATTO740_name = emission_spectra[43]

dye_names = np.array(
    [ATTO390_name, ATTO488_name, ATTO550_name, ATTO647N_name, ATTO740_name]
)

In [ ]:
data_saving_folder = "/home/jbeckwith/Documents/Dropbox/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Data/Simulation/20241023/data"
saving_names = np.array(
    [
        "ATTO390_fittesting",
        "ATTO488_fittesting",
        "ATTO550_fittesting",
        "ATTO647N_fittesting",
        "ATTO740_fittesting",
    ]
)

In [ ]:
image_size = 26

from Camera_QE import getpixelefficiency

gpe = getpixelefficiency.GPE()
R, G, B, wavelength = gpe.getpixelefficiency("Camera_QE/CS505CU_QE.csv")
masks = MSF.MF.get_masks(MSF.mosaic_unit, image_size, image_size)
masks_3d = np.dstack([masks["maskR"], masks["maskG"], masks["maskB"]])
wavelength = wavelength
absolute_QYs = np.vstack([B, G, R])
camera_calibration = {}
camera_calibration["gain"] = gain[:image_size, :image_size]
camera_calibration["offset"] = offset[:image_size, :image_size]
camera_calibration["variance"] = variance[:image_size, :image_size]
camera_calibration["readnoise"] = readnoise[:image_size, :image_size]
camera_calibration["rqe"] = rqe[:image_size, :image_size]

In [ ]:
n_photon_lspace = np.linspace(500, 20000, 1000)
n_bootstrap = 500
background_photons = 0
pixel_size = 69
NA = 1.49

In [ ]:
max = pixel_size * image_size
x0 = np.full(n_bootstrap, max / 2) + np.random.normal(size=n_bootstrap) * (
    pixel_size / 2
)
y0 = np.full(n_bootstrap, max / 2) + np.random.normal(size=n_bootstrap) * (
    pixel_size / 2
)

In [ ]:
for k, dye in enumerate(dye_names):
    dyes = np.zeros([1, len(wavelength)])

    separator = "\t"
    skip_rows = 1

    data = pl.read_csv(
        os.path.join(folder, dye), separator=separator, skip_rows=skip_rows
    )
    d_wl = data[:, 0].to_numpy()
    d_em = data[:, 1].to_numpy()
    ATTO550 = np.interp(x=wavelength, xp=d_wl, fp=d_em)
    ATTO550 = ATTO550 / np.sum(ATTO550)
    dyes[0, :] = ATTO550

    average_emission_wavelength = np.trapz(
        y=wavelength * (dyes.T / np.trapz(x=wavelength, y=dyes)).T, x=wavelength
    )
    sigma_PSF = PSF.sigma_PSF(float(average_emission_wavelength[0]), NA)

    parameters = np.array(["xc", "yc", "A", "sigma", "b", "R", "G", "B"])
    dye_pixel_efficiency = np.dot(dyes, absolute_QYs.T)
    dye_fit_expectation = dye_pixel_efficiency[0][::-1] / np.sum(
        dye_pixel_efficiency[0][::-1]
    )

    x0y0 = {}

    fit_RMSE_mean = np.zeros([len(parameters), len(n_photon_lspace)])
    fit_RMSE_std = np.zeros([len(parameters), len(n_photon_lspace)])
    start = time.time()

    expected_parameters = np.array(
        [
            max / 2,
            max / 2,
            sigma_PSF,
            0,
            dye_fit_expectation[0],
            dye_fit_expectation[1],
            dye_fit_expectation[2],
        ]
    )
    real_params = pl.DataFrame(
        data=np.expand_dims(expected_parameters, 0),
        schema=list(parameters[[0, 1, 3, 4, 5, 6, 7]]),
    )
    real_params.write_csv(
        os.path.join(data_saving_folder, saving_names[k] + "_input_parameters.csv")
    )

    x0 = np.full(n_bootstrap, max / 2) + np.random.normal(size=n_bootstrap) * (
        pixel_size / 2
    )
    y0 = np.full(n_bootstrap, max / 2) + np.random.normal(size=n_bootstrap) * (
        pixel_size / 2
    )

    for i, n_photon in enumerate(n_photon_lspace):
        expected_parameters = np.array(
            [
                max / 2,
                max / 2,
                n_photon,
                sigma_PSF,
                0,
                dye_fit_expectation[0],
                dye_fit_expectation[1],
                dye_fit_expectation[2],
            ]
        )
        fit_values = np.zeros([len(parameters), n_bootstrap])
        n_photons = {}
        n_photons["dye"] = n_photon

        for j in np.arange(n_bootstrap):
            x0y0["dye"] = np.array([[x0[j]], [y0[j]]])
            ground_truth, bayer_image = MSF.gen_camera_images(
                camera_calibration,
                wavelength,
                absolute_QYs,
                dyes,
                n_photons,
                x0y0,
                background_photons=background_photons,
                NA=NA,
                pixel_size=pixel_size,
            )
            del x0y0["dye"]
            photoelectron_data = np.divide(
                np.divide(
                    np.subtract(bayer_image, camera_calibration["offset"]),
                    camera_calibration["gain"],
                ),
                camera_calibration["rqe"],
            )
            erd = sCMOS.var_weighted_uniform_filter(
                photoelectron_data, camera_calibration["variance"], 4
            )
            erd[erd < 0] = 0
            erd = erd + 1
            error_map = np.add(erd, np.square(camera_calibration["readnoise"]))
            weights_map = np.power(error_map, -2)

            xc_ig, yc_ig = np.unravel_index(
                np.argmax(
                    sCMOS.var_weighted_uniform_filter(
                        photoelectron_data, camera_calibration["variance"], 4
                    )
                ),
                photoelectron_data.shape,
            )
            A = np.sum(
                np.abs(
                    sCMOS.var_weighted_uniform_filter(
                        photoelectron_data, camera_calibration["variance"], 4
                    )
                )
            )
            sigma = 3
            b = 0
            initial_guess = np.array([xc_ig, yc_ig, A, sigma, b, 0.5, 0.5, 0.5])
            result = AF.WLS_fit_LM(
                photoelectron_data,
                initial_guess,
                masks=masks_3d,
                weights=weights_map,
                display=False,
            )
            if result.status < 0:
                fit_values[:, j] = np.full_like(result.x, np.NAN)
            else:
                fit_values[:, j] = result.x

        print(
            "Analysed photon flux {}/{}    Time elapsed: {:.3f} min".format(
                i + 1, len(n_photon_lspace), (time.time() - start) / 60.0
            ),
            end="\r",
            flush=True,
        )
        fit_values[[0, 1, 3], :] = fit_values[[0, 1, 3], :] * pixel_size

        # fig, axs = plotter.two_column_plot(nrows=2, ncolumns=2, widthratio=[1,1], heightratio=[1,1])

        # axs[0,0] = plotter.histogram_plot(axs=axs[0,0], data=np.sqrt(np.square(fit_values[0, :] - y0)), bins=np.histogram_bin_edges(np.sqrt(np.square(fit_values[0, :] - y0)), bins='fd'), xaxislabel='distance error / nm')
        # axs[0,1] = plotter.histogram_plot(axs=axs[0,1], data=fit_values[2, :], bins=np.histogram_bin_edges(fit_values[2, :], bins='fd'), xaxislabel='Amplitude')
        # axs[1,0] = plotter.histogram_plot(axs=axs[1,0], data=fit_values[3, :], bins=np.histogram_bin_edges(fit_values[3, :], bins='fd'), xaxislabel=r'$\sigma$')
        # axs[1,1] = plotter.histogram_plot(axs=axs[1,1], data=np.sqrt(np.square(fit_values[5, :] -  expected_parameters[5])), bins=np.histogram_bin_edges(np.sqrt(np.square(fit_values[5, :] -  expected_parameters[5])), bins='fd'), xaxislabel=r'R error')
        # plt.show(block=False)

        for param in np.arange(len(parameters)):
            if param == 0:
                fit_RMSE_mean[param, i] = np.nanmean(
                    np.sqrt(np.square((fit_values[param, :] - y0)))
                )
                fit_RMSE_std[param, i] = np.nanstd(
                    np.sqrt(np.square((fit_values[param, :] - y0)))
                )
            elif param == 1:
                fit_RMSE_mean[param, i] = np.nanmean(
                    np.sqrt(np.square((fit_values[param, :] - x0)))
                )
                fit_RMSE_std[param, i] = np.nanstd(
                    np.sqrt(np.square((fit_values[param, :] - x0)))
                )
            else:
                fit_RMSE_mean[param, i] = np.nanmean(
                    np.sqrt(
                        np.square((fit_values[param, :] - expected_parameters[param]))
                    )
                )
                fit_RMSE_std[param, i] = np.nanstd(
                    np.sqrt(
                        np.square((fit_values[param, :] - expected_parameters[param]))
                    )
                )

    parameters_tosave = np.array(
        ["n_photons", "xc", "yc", "A", "sigma", "b", "R", "G", "B"]
    )
    means = pl.DataFrame(
        data=np.vstack([n_photon_lspace, fit_RMSE_mean]), schema=list(parameters_tosave)
    )
    stds = pl.DataFrame(
        data=np.vstack([n_photon_lspace, fit_RMSE_std]), schema=list(parameters_tosave)
    )
    means.write_csv(
        os.path.join(
            data_saving_folder,
            saving_names[k] + "_69nmpixelsize_RMSE_mean_VUxyestimation_LM.csv",
        )
    )
    stds.write_csv(
        os.path.join(
            data_saving_folder,
            saving_names[k] + "_69nmpixelsize_RMSE_std_VUxyestimation_LM.csv",
        )
    )

In [ ]:
import polars as pl

In [ ]:
folder = "/home/jbeckwith/Documents/Dropbox/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Data/Simulation/20241023/data"
fittesting_ATTO390_VU_RN = pl.read_csv(
    os.path.join(
        folder, "ATTO390_fittesting_69nmpixelsize_RMSE_mean_VUxyestimation_LM.csv"
    )
)
fittesting_ATTO488_VU_RN = pl.read_csv(
    os.path.join(
        folder, "ATTO488_fittesting_69nmpixelsize_RMSE_mean_VUxyestimation_LM.csv"
    )
)
fittesting_ATTO550_VU_RN = pl.read_csv(
    os.path.join(
        folder, "ATTO550_fittesting_69nmpixelsize_RMSE_mean_VUxyestimation_LM.csv"
    )
)
fittesting_ATTO647N_VU_RN = pl.read_csv(
    os.path.join(
        folder, "ATTO647N_fittesting_69nmpixelsize_RMSE_mean_VUxyestimation_LM.csv"
    )
)
fittesting_ATTO740_VU_RN = pl.read_csv(
    os.path.join(
        folder, "ATTO740_fittesting_69nmpixelsize_RMSE_mean_VUxyestimation_LM.csv"
    )
)

fittest_VU_RN = (
    fittesting_ATTO390_VU_RN
    + fittesting_ATTO488_VU_RN
    + fittesting_ATTO550_VU_RN
    + fittesting_ATTO647N_VU_RN
    + fittesting_ATTO740_VU_RN
) / 5

In [ ]:
folder = "/home/jbeckwith/Documents/Dropbox/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Data/Simulation/20241023/data"
fittesting_ATTO390_VU_RN_std = pl.read_csv(
    os.path.join(
        folder, "ATTO390_fittesting_69nmpixelsize_RMSE_std_VUxyestimation_LM.csv"
    )
)
fittesting_ATTO488_VU_RN_std = pl.read_csv(
    os.path.join(
        folder, "ATTO488_fittesting_69nmpixelsize_RMSE_std_VUxyestimation_LM.csv"
    )
)
fittesting_ATTO550_VU_RN_std = pl.read_csv(
    os.path.join(
        folder, "ATTO550_fittesting_69nmpixelsize_RMSE_std_VUxyestimation_LM.csv"
    )
)
fittesting_ATTO647N_VU_RN_std = pl.read_csv(
    os.path.join(
        folder, "ATTO647N_fittesting_69nmpixelsize_RMSE_std_VUxyestimation_LM.csv"
    )
)
fittesting_ATTO740_VU_RN_std = pl.read_csv(
    os.path.join(
        folder, "ATTO740_fittesting_69nmpixelsize_RMSE_std_VUxyestimation_LM.csv"
    )
)

fittest_VU_RN_std = (
    fittesting_ATTO390_VU_RN_std
    + fittesting_ATTO488_VU_RN_std
    + fittesting_ATTO550_VU_RN_std
    + fittesting_ATTO647N_VU_RN_std
    + fittesting_ATTO740_VU_RN_std
) / 5

In [ ]:
folder = "/home/jbeckwith/Documents/Dropbox/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Data/Simulation/20241023/data"
fittesting_ATTO390_VU_RN_TRF = pl.read_csv(
    os.path.join(
        folder, "ATTO390_fittesting_69nmpixelsize_RMSE_mean_VUxyestimation_TRF.csv"
    )
)
fittesting_ATTO488_VU_RN_TRF = pl.read_csv(
    os.path.join(
        folder, "ATTO488_fittesting_69nmpixelsize_RMSE_mean_VUxyestimation_TRF.csv"
    )
)
fittesting_ATTO550_VU_RN_TRF = pl.read_csv(
    os.path.join(
        folder, "ATTO550_fittesting_69nmpixelsize_RMSE_mean_VUxyestimation_TRF.csv"
    )
)
fittesting_ATTO647N_VU_RN_TRF = pl.read_csv(
    os.path.join(
        folder, "ATTO647N_fittesting_69nmpixelsize_RMSE_mean_VUxyestimation_TRF.csv"
    )
)
fittesting_ATTO740_VU_RN_TRF = pl.read_csv(
    os.path.join(
        folder, "ATTO740_fittesting_69nmpixelsize_RMSE_mean_VUxyestimation_TRF.csv"
    )
)

fittest_VU_RN_TRF = (
    fittesting_ATTO390_VU_RN_TRF
    + fittesting_ATTO488_VU_RN_TRF
    + fittesting_ATTO550_VU_RN_TRF
    + fittesting_ATTO647N_VU_RN_TRF
    + fittesting_ATTO740_VU_RN_TRF
) / 5

In [ ]:
folder = "/home/jbeckwith/Documents/Dropbox/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Data/Simulation/20241023/data"
fittesting_ATTO390_VU_RN_TRF_std = pl.read_csv(
    os.path.join(
        folder, "ATTO390_fittesting_69nmpixelsize_RMSE_std_VUxyestimation_TRF.csv"
    )
)
fittesting_ATTO488_VU_RN_TRF_std = pl.read_csv(
    os.path.join(
        folder, "ATTO488_fittesting_69nmpixelsize_RMSE_std_VUxyestimation_TRF.csv"
    )
)
fittesting_ATTO550_VU_RN_TRF_std = pl.read_csv(
    os.path.join(
        folder, "ATTO550_fittesting_69nmpixelsize_RMSE_std_VUxyestimation_TRF.csv"
    )
)
fittesting_ATTO647N_VU_RN_TRF_std = pl.read_csv(
    os.path.join(
        folder, "ATTO647N_fittesting_69nmpixelsize_RMSE_std_VUxyestimation_TRF.csv"
    )
)
fittesting_ATTO740_VU_RN_TRF_std = pl.read_csv(
    os.path.join(
        folder, "ATTO740_fittesting_69nmpixelsize_RMSE_std_VUxyestimation_TRF.csv"
    )
)

fittest_VU_RN_TRF_std = (
    fittesting_ATTO390_VU_RN_TRF_std
    + fittesting_ATTO488_VU_RN_TRF_std
    + fittesting_ATTO550_VU_RN_TRF
    + fittesting_ATTO647N_VU_RN_TRF_std
    + fittesting_ATTO740_VU_RN_TRF
) / 5

In [ ]:
fig, axs = plotter.two_column_plot(ncolumns=2, widthratio=[1, 1])

axs[0] = plotter.scatter_plot(
    axs=axs[0],
    x=fittest_VU_RN["n_photons"].to_numpy(),
    y=0.5 * (fittest_VU_RN["xc"].to_numpy() + fittest_VU_RN["yc"].to_numpy()),
    xaxislabel="number of photons",
    yaxislabel="localisation RMSE / nm",
    edgecolor="r",
    label="Levenberg–Marquardt",
    s=1,
)

axs[0] = plotter.scatter_plot(
    axs=axs[0],
    x=fittest_VU_RN_TRF["n_photons"].to_numpy(),
    y=0.5 * (fittest_VU_RN_TRF["xc"].to_numpy() + fittest_VU_RN_TRF["yc"].to_numpy()),
    xaxislabel="number of photons",
    yaxislabel="localisation RMSE / nm",
    edgecolor="k",
    label="Constrained Region",
    s=1,
)

axs[1] = plotter.scatter_plot(
    axs=axs[1],
    x=fittesting_ATTO390_VU_RN["n_photons"].to_numpy(),
    y=(100.0 / 3)
    * (
        fittest_VU_RN["R"].to_numpy()
        + fittest_VU_RN["G"].to_numpy()
        + fittest_VU_RN["B"].to_numpy()
    ),
    xaxislabel="number of photons",
    yaxislabel="colour RMSE / normalised",
    edgecolor="r",
    s=1,
)

axs[1] = plotter.scatter_plot(
    axs=axs[1],
    x=fittest_VU_RN_TRF["n_photons"].to_numpy(),
    y=(100.0 / 3)
    * (
        fittest_VU_RN_TRF["R"].to_numpy()
        + fittest_VU_RN_TRF["G"].to_numpy()
        + fittest_VU_RN_TRF["B"].to_numpy()
    ),
    xaxislabel="number of photons",
    yaxislabel="colour RMSE / %",
    edgecolor="k",
    s=1,
)

axs[0].legend(loc="best")
axs[0].set_xscale("log")
axs[0].set_ylim([0.7, 50])
axs[0].set_yscale("log")

axs[1].set_xscale("log")
axs[1].set_ylim([0.4, 50])
axs[1].set_yscale("log")
plt.tight_layout()

savefolder = "/home/jbeckwith/Documents/Dropbox/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Talks+Posters/Subgroup/20241114/fig/multicolour_camera/algorithm"

plt.savefig(
    os.path.join(savefolder, "Overall_Algorithm_Reliability_LM+TRF.svg"),
    format="svg",
    dpi=600,
)
plt.show()

In [ ]:
fig, axs = plotter.two_column_plot(ncolumns=2, widthratio=[1, 1])

axs[0] = plotter.scatter_plot(
    axs=axs[0],
    x=fittest_VU_RN_std["n_photons"].to_numpy(),
    y=0.5 * (fittest_VU_RN_std["xc"].to_numpy() + fittest_VU_RN_std["yc"].to_numpy()),
    xaxislabel="number of photons",
    yaxislabel="localisation $\sigma$ / nm",
    edgecolor="r",
    label="Levenberg–Marquardt",
    s=1,
)

axs[0] = plotter.scatter_plot(
    axs=axs[0],
    x=fittest_VU_RN_TRF_std["n_photons"].to_numpy(),
    y=0.5
    * (fittest_VU_RN_TRF_std["xc"].to_numpy() + fittest_VU_RN_TRF_std["yc"].to_numpy()),
    xaxislabel="number of photons",
    yaxislabel="localisation $\sigma$ / nm",
    edgecolor="k",
    label="Constrained Region",
    s=1,
)


axs[1] = plotter.scatter_plot(
    axs=axs[1],
    x=fittest_VU_RN_TRF_std["n_photons"].to_numpy(),
    y=(100.0 / 3)
    * (
        fittest_VU_RN_TRF_std["R"].to_numpy()
        + fittest_VU_RN_TRF_std["G"].to_numpy()
        + fittest_VU_RN_TRF_std["B"].to_numpy()
    ),
    xaxislabel="number of photons",
    yaxislabel="colour $\sigma$ / %",
    edgecolor="k",
    s=1,
)

axs[1] = plotter.scatter_plot(
    axs=axs[1],
    x=fittest_VU_RN_std["n_photons"].to_numpy(),
    y=(100.0 / 3)
    * (
        fittest_VU_RN_std["R"].to_numpy()
        + fittest_VU_RN_std["G"].to_numpy()
        + fittest_VU_RN_std["B"].to_numpy()
    ),
    xaxislabel="number of photons",
    yaxislabel="colour $\sigma$ / %",
    edgecolor="r",
    s=1,
)

axs[0].legend(loc="best")
# axs[0].set_xscale('log')
axs[0].set_ylim([1, 11])
# axs[0].set_yscale('log')

# axs[1].set_xscale('log')
axs[1].set_ylim([0.1, 1000])
axs[1].set_yscale("log")
plt.tight_layout()

savefolder = "/home/jbeckwith/Documents/Dropbox/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Talks+Posters/Subgroup/20241114/fig/multicolour_camera/algorithm"

plt.savefig(
    os.path.join(savefolder, "Overall_Algorithm_Reliability_LM+TRF_std.svg"),
    format="svg",
    dpi=600,
)
plt.show()

In [ ]:
overall_fit_array = [
    fittesting_ATTO390_VU_RN,
    fittesting_ATTO488_VU_RN,
    fittesting_ATTO550_VU_RN,
    fittesting_ATTO647N_VU_RN,
    fittesting_ATTO740_VU_RN,
]

overall_fit_array_TRF = [
    fittesting_ATTO390_VU_RN_TRF,
    fittesting_ATTO488_VU_RN_TRF,
    fittesting_ATTO550_VU_RN_TRF,
    fittesting_ATTO647N_VU_RN_TRF,
    fittesting_ATTO740_VU_RN_TRF,
]

cs = ["#006a9e", "#00b276", "#f55200", "#d80000", "#8b0000"]
string_array = ["ATTO390", "ATTO488", "ATTO550", "ATTO647N", "ATTO740"]
point_type = ["o", "v", "s", "X", "d"]

fig, axs = plotter.two_column_plot(ncolumns=2, widthratio=[1, 1])

for i, array in enumerate(overall_fit_array):
    array_TRF = overall_fit_array_TRF[i]
    axs[0] = plotter.scatter_plot(
        axs=axs[0],
        x=array["n_photons"].to_numpy(),
        y=0.5 * (array["xc"].to_numpy() + array["yc"].to_numpy()),
        xaxislabel="number of photons",
        yaxislabel="localisation RMSE / nm",
        label=string_array[i],
        edgecolor=cs[i],
        marker=point_type[i],
        s=2,
    )

    axs[1] = plotter.scatter_plot(
        axs=axs[1],
        x=array_TRF["n_photons"].to_numpy(),
        y=0.5 * (array_TRF["xc"].to_numpy() + array_TRF["yc"].to_numpy()),
        xaxislabel="number of photons",
        yaxislabel="",
        edgecolor=cs[i],
        marker=point_type[i],
        s=2,
    )

axs[0].legend(loc="best", fontsize=6)
axs[0].set_xscale("log")
axs[0].set_ylim([0.7, 25])
axs[0].set_yscale("log")

axs[1].set_xscale("log")
axs[1].set_ylim([0.7, 25])
axs[1].set_yscale("log")
plt.tight_layout()

savefolder = "/home/jbeckwith/Documents/Dropbox/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Talks+Posters/Subgroup/20241114/fig/multicolour_camera/algorithm"

plt.savefig(
    os.path.join(savefolder, "Algorithm_Reliability_vs_Colour_LMTRF_Sigma.svg"),
    format="svg",
    dpi=600,
)
plt.show()

In [ ]:
overall_fit_array = [
    fittesting_ATTO390_VU_RN,
    fittesting_ATTO488_VU_RN,
    fittesting_ATTO550_VU_RN,
    fittesting_ATTO647N_VU_RN,
    fittesting_ATTO740_VU_RN,
]

overall_fit_array_TRF = [
    fittesting_ATTO390_VU_RN_TRF,
    fittesting_ATTO488_VU_RN_TRF,
    fittesting_ATTO550_VU_RN_TRF,
    fittesting_ATTO647N_VU_RN_TRF,
    fittesting_ATTO740_VU_RN_TRF,
]

cs = ["#006a9e", "#00b276", "#f55200", "#d80000", "#8b0000"]
string_array = ["ATTO390", "ATTO488", "ATTO550", "ATTO647N", "ATTO740"]
point_type = ["o", "v", "s", "X", "d"]

fig, axs = plotter.two_column_plot(ncolumns=2, widthratio=[1, 1])

for i, array in enumerate(overall_fit_array):
    array_TRF = overall_fit_array_TRF[i]
    axs[0] = plotter.scatter_plot(
        axs=axs[0],
        x=array["n_photons"].to_numpy(),
        y=(100.0 / 3)
        * (array["R"].to_numpy() + array["G"].to_numpy() + array["B"].to_numpy()),
        xaxislabel="number of photons",
        yaxislabel="colour RMSE / %",
        label=string_array[i],
        edgecolor=cs[i],
        marker=point_type[i],
        s=2,
    )

    axs[1] = plotter.scatter_plot(
        axs=axs[1],
        x=array_TRF["n_photons"].to_numpy(),
        y=(100.0 / 3)
        * (
            array_TRF["R"].to_numpy()
            + array_TRF["G"].to_numpy()
            + array_TRF["B"].to_numpy()
        ),
        yaxislabel="",
        xaxislabel="number of photons",
        edgecolor=cs[i],
        marker=point_type[i],
        s=2,
    )

axs[0].legend(loc="best", fontsize=6)
axs[0].set_xscale("log")
axs[0].set_ylim([0.4, 25])
axs[0].set_yscale("log")

axs[1].set_xscale("log")
axs[1].set_ylim([0.4, 25])
axs[1].set_yscale("log")
plt.tight_layout()

savefolder = "/home/jbeckwith/Documents/Dropbox/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Talks+Posters/Subgroup/20241114/fig/multicolour_camera/algorithm"

plt.savefig(
    os.path.join(savefolder, "Algorithm_Reliability_vs_Colour_LMTRF_RGB.svg"),
    format="svg",
    dpi=600,
)
plt.show()

In [ ]:
from src import PlottingFunctions

plotter = PlottingFunctions.Plotter()

import matplotlib.pyplot as plt
import mpltern
from matplotlib.ticker import MultipleLocator, AutoMinorLocator

from Camera_QE import getpixelefficiency
from copy import copy

gpe = getpixelefficiency.GPE()
R, G, B, wavelength = gpe.getpixelefficiency("Camera_QE/CS505CU_QE.csv")

folder = "Spectra/Em/Dyes/"
emission_spectra = np.sort(os.listdir(folder))
emission_spectra = [
    x
    for x in emission_spectra
    if ("ATTO390" in x)
    or ("ATTO488" in x)
    or ("ATTO550" in x)
    or ("ATTO647N" in x)
    or ("ATTO740" in x)
]
meanwl = np.zeros(len(emission_spectra))
RGB = np.zeros([3, len(emission_spectra)])

for i, file in enumerate(emission_spectra):
    if "ATTO" in file:
        separator = "\t"
        skip_rows = 1
    else:
        separator = ","
        skip_rows = 0
    data = pl.read_csv(
        os.path.join(folder, file), separator=separator, skip_rows=skip_rows
    )
    d_wl = data[:, 0].to_numpy()
    d_em = data[:, 1].to_numpy()
    d_em = d_em / np.trapz(x=d_wl, y=d_em)
    meanwl[i] = np.trapz(x=d_wl, y=d_em * d_wl)
    RGB[0, i] = np.sum(np.interp(x=d_wl, xp=wavelength, fp=R) * d_em)
    RGB[1, i] = np.sum(np.interp(x=d_wl, xp=wavelength, fp=G) * d_em)
    RGB[2, i] = np.sum(np.interp(x=d_wl, xp=wavelength, fp=B) * d_em)

RGB = RGB / np.sum(RGB, axis=0)

Rd = RGB[0, :]
Gd = RGB[1, :]
Bd = RGB[2, :]

fig, ax = plotter.two_column_plot()

ax = fig.add_subplot(1, 1, 1, projection="ternary")

# Color ticks, grids, tick-labels
ax.taxis.set_tick_params(tick2On=True, colors="r", grid_color="r", which="both")
ax.taxis.label.set_color("r")
ax.spines["lside"].set_color("r")

ax.laxis.set_tick_params(tick2On=True, colors="g", grid_color="g", which="both")
ax.laxis.label.set_color("g")
ax.spines["rside"].set_color("g")

ax.raxis.set_tick_params(tick2On=True, colors="b", grid_color="b", which="both")
ax.raxis.label.set_color("b")
ax.spines["tside"].set_color("b")


ax.set_tlabel(r"R")
ax.set_llabel(r"G")
ax.set_rlabel(r"B")

ax.grid(lw=0.5, alpha=0.25, ls="--", which="both", axis="both")

c = np.vstack([RGB, np.ones_like(Rd)])

pc = ax.scatter(
    Rd,
    Gd,
    Bd,
    s=10,
    facecolors=c.T,
    edgecolors="None",
    lw=0,
    marker="o",
    label="true position",
)

In [ ]:
from mpltern.datasets import get_shanon_entropies

t, l, r, v = get_shanon_entropies()

In [ ]:
plt.plot(v)

In [ ]:
np.max(r)

In [ ]:
fig = plt.figure(figsize=(10.8, 4.8))
fig.subplots_adjust(left=0.075, right=0.85, wspace=0.3)

# These values are for controlling the color-bar scale, and here they are
# explicitly given just to make the same color-bar scale for all the plots.
# In general, you may not need to explicitly specify them.
vmin = 0.0
vmax = 1.2
levels = np.linspace(vmin, vmax, 7)

ax = fig.add_subplot(1, 2, 1, projection="ternary")
cs = ax.tricontour(t, l, r, v, levels=levels)
ax.clabel(cs)
ax.set_title("tricontour")

cax = ax.inset_axes([1.05, 0.1, 0.05, 0.9], transform=ax.transAxes)
colorbar = fig.colorbar(cs, cax=cax)
colorbar.set_label("Entropy", rotation=270, va="baseline")

ax = fig.add_subplot(1, 2, 2, projection="ternary")
cs = ax.tricontourf(t, l, r, v, levels=levels)
ax.set_title("tricontourf")

cax = ax.inset_axes([1.05, 0.1, 0.05, 0.9], transform=ax.transAxes)
colorbar = fig.colorbar(cs, cax=cax)
colorbar.set_label("Entropy", rotation=270, va="baseline")